# Merge EPA Water Quality/Climate dataset with Agriculture

In [ ]:
import pandas as pd
import numpy as np

# Load EPA-Climate merged data
epa_climate_df = pd.read_csv('../../data/tabular/merged/epa-climate-merged.csv')

# Load Agriculture data
ag_df = pd.read_csv('../../data/tabular/agricultural/clean/usdaNass-agriculture-clean.csv')

In [ ]:
print("Preparing data for merge...")
# Ensure date columns are datetime
epa_climate_df['day_dt'] = pd.to_datetime(epa_climate_df['day'])
epa_climate_df['year'] = epa_climate_df['day_dt'].dt.year
epa_climate_df['state'] = 'IOWA' 

print("Pivoting agriculture data to wide format...")
# Clean up Data Item names for column headers
ag_df['data_item_short'] = ag_df['data_item'].str.replace(', MEASURED IN .*', '', regex=True)

# Pivot
ag_pivot = ag_df.pivot_table(
    index=['year', 'period', 'state'],
    columns='data_item_short',
    values='value',
    aggfunc='first'
).reset_index()

# Create month mapping for Ag data
months = ['JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN', 'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC']
ag_pivot['month'] = ag_pivot['period'].where(ag_pivot['period'].isin(months))

# For dates in epa_climate, get the month
epa_climate_df['month'] = epa_climate_df['day_dt'].dt.strftime('%b').str.upper()

In [ ]:
print("Merging on Year, Month, and State...")
merged_df = epa_climate_df.merge(
    ag_pivot[ag_pivot['month'].notna()],
    left_on=['year', 'month', 'state'],
    right_on=['year', 'month', 'state'],
    how='left'
)

print(f"Final merged shape: {merged_df.shape}")

In [ ]:
# Save the result
output_path = '../../data/tabular/merged/epa-climate-agriculture-merged.csv'
merged_df.to_csv(output_path, index=False)
print(f"Saved merged data to {output_path}")